# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant
# For visualization
!pip install matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print(f"Name: {getattr(dataset.metadata, 'name', None)}\nDescription: {getattr(dataset.metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect all the record sets, their `@id` values, and their field structure.

In [ ]:
# List available record sets in the dataset using their @id
record_sets = getattr(dataset.metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rec in record_sets:
        # Each entry is expected to be a dict or object with an '@id'
        rec_id = getattr(rec, '@id', rec.get('@id')) if isinstance(rec, dict) else rec
        rec_name = getattr(rec, 'name', rec.get('name', '')) if isinstance(rec, dict) else ''
        print(f"- RecordSet: {rec_id}, Name: {rec_name}")
        # List fields for each record set
        fields = getattr(rec, 'field', rec.get('field', [])) if isinstance(rec, dict) else []
        if fields:
            print("  Fields:")
            for fld in fields:
                fld_id = getattr(fld, '@id', fld.get('@id')) if isinstance(fld, dict) else fld
                fld_name = getattr(fld, 'name', fld.get('name', '')) if isinstance(fld, dict) else ''
                print(f"    - {fld_id} ({fld_name})")
        else:
            print("  No fields found.")

If the dataset does not declare explicit record sets in the top-level `recordSet`, the mlcroissant library should be able to list them via the `records()` iterator.

Let's enumerate all possible record sets by checking the available data sets:

In [ ]:
# Try automatic detection of record sets from the dataset
available_record_sets = dataset.list_record_sets()
print("Available Record Sets (by @id):")
for rec_set in available_record_sets:
    print(f"- {rec_set}")

# For demonstration, show the field structure of the first record set
if available_record_sets:
    first_rs = available_record_sets[0]
    rs_obj = dataset.get_record_set(first_rs)
    print(f"\nFields in RecordSet '{first_rs}':")
    if hasattr(rs_obj, 'field'):
        for f in rs_obj.field:
            print(f"- {getattr(f, '@id', None)} (name: {getattr(f, 'name', '')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select all detected record set @id values
record_sets = available_record_sets
dataframes = {}
for record_set in record_sets:
    print(f"Extracting records for record set '@id': {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"  Fields (columns): {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records found.")

# Select the main table for further analysis (use the first if uncertain)
main_record_set = record_sets[0] if record_sets else None
if main_record_set:
    print(f"\nColumns in main record set '{main_record_set}':")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below we:
- Select a numeric field (by its `@id`/column name).
- Filter records with a threshold.
- Normalize the field.
- Group by a categorical field (if present).

Replace `<numeric_field>` and `<group_field>` with actual values from your data exploration above.

In [ ]:
# Choose numeric and group fields by @id (adjust as per the actual field names found previously)
df = dataframes[main_record_set]
print("Available columns in main record set:", df.columns.tolist())

# Attempt to auto-select a numeric column
numeric_field = None
for col in df.columns:
    # Try to find a likely numeric column
    if df[col].dtype.kind in 'iufc' or ('amount' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or 'coverage' in col.lower()):
        numeric_field = col
        break
# Fallback: just pick the first one
if numeric_field is None and len(df.columns) > 1:
    numeric_field = df.columns[1]

print(f"Using numeric field: {numeric_field}")

if numeric_field:
    # Convert col to numeric if not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.5)  # Median as threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to find a group/categorical field
    group_field = None
    for col in df.columns:
        if col != numeric_field and (df[col].dtype == object or df[col].dtype.name == 'category'):
            group_field = col
            break
    print(f"Grouping field: {group_field}")
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of the selected numeric field and visualize grouped means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field:
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='steelblue')
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # If we have a grouping field and group means
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 6))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.ylabel(f"Average {numeric_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library. We listed record sets (identified by `@id`), dynamically extracted fields, filtered and normalized a numeric field, grouped and visualized results. You may extend this analysis further by exploring other record sets or performing domain-specific analysis on protein abundance and modifications.

For best results, always reference data entities (record sets, fields, columns) by their `@id`. Refer to the Croissant schema for authoritative field identifiers and documentation.